# CALM — Adaptive Multimodal Wildfire Monitoring

Demo hệ thống AI giám sát cháy rừng: Planning → Routing → Execution → Evaluation.

- **Plan-driven:** PlanningAgent → RouterAgent → ExecutionAgent
- **Agents:** DataKnowledge, Prediction (SeasFire/heuristic), QA, RSEN, Evaluator

## 1. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")

_root = Path.cwd()
_src = _root / "src" if (_root / "src").exists() else _root / "calm" / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from calm.utils.env_loader import load_env
load_env(_root / ".env")
if (_root / "calm").exists():
    load_env(_root / "calm" / ".env")

print("Setup OK")

In [ ]:
import os

if os.environ.get("OPENROUTER_API_KEY"):
    from langchain_openrouter import ChatOpenRouter
    llm = ChatOpenRouter(
        model=os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o"),
        api_key=os.environ["OPENROUTER_API_KEY"],
        temperature=0.0,
    )
elif os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
else:
    raise ValueError("Set OPENAI_API_KEY or OPENROUTER_API_KEY in .env")

print("LLM ready")

## 2. Planning Agent

URSA 3-node: Generator → Reflector → Formalizer. Output: plan JSON với step_id, action, agent, prompt.

In [ ]:
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = planner.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Steps:", len(plan))
for i, s in enumerate(plan, 1):
    print(f"  {i}. {s.get('step_id')} | {s.get('action')} | {s.get('agent')}")
    if s.get("prompt"):
        print(f"     prompt: {s['prompt'][:80]}...")

## 3. Router Agent

Xác định task_type (qa | prediction | hybrid) từ plan + query.

In [ ]:
from calm.agents.router_agent import RouterAgent

router = RouterAgent(llm=llm, config={})
routing = router.route(query, plan)
print("Task type:", routing.task_type)
print("Confidence:", routing.confidence)
print("Required artifacts:", routing.required_artifacts)

## 4. DataKnowledgeAgent

Collect (GEE, CDS, Web, ArXiv) → Extract → Retrieve (ChromaDB). GeocodingTool chuyển địa chỉ văn bản → lat, lon.

In [ ]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.memory_agent import MemoryAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool
from calm.tools.geocoding import GeocodingTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
geocoding = GeocodingTool(safety_checker=safety, config={})
tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None, "geocoding": geocoding}

memory = MemoryAgent(long_term_store=ChromaMemoryStore(collection_name="calm_demo", persist_directory=".chroma", k=3))
data_agent = DataKnowledgeAgent(llm=llm, tools=tools, memory_store=memory, config={"dedup_check": False})

params = {"location": "California, USA", "time_range": {"start": "2024-01-01", "end": "2024-12-31"}}
ret = data_agent.retrieve("wildfire causes in California 2024", parameters=params)
print("Retrieved:", len(ret.get("retrieved_data", [])), "sources")
print("Dedup hit:", ret.get("dedup", False))

## 5. RSEN — Validation

Weather Analyst + Geo Analyst (song song) → Ops Coordinator → Plausible/Implausible.

In [ ]:
from calm.agents.rsen_module import RSENModule

rsen = RSENModule(llm=llm, memory_store=memory, k=3)
val = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2},
    spatial_data={"fuel_type": "Shrubland", "slope": 25},
)
print("Decision:", val.get("validation_decision"))
print("Rationale:", (val.get("final_rationale") or "")[:200]+"...")

## 6. Wildfire QA Agent

Evidence Evaluator → retrieve → trả lời với citations.

In [ ]:
from calm.agents.qa_agent import WildfireQAAgent

qa = WildfireQAAgent(llm=llm, data_agent=data_agent, web_search_tool=web, memory_store=memory, config={"evidence_threshold": 0.65})
qa_result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = qa_result.get("final_output") or {}
print("Answer:", (out.get("answer", ""))[:300])
print("Citations:", out.get("citations", [])[:2])
print("Confidence:", out.get("confidence"))

## 7. Full Pipeline — CALMOrchestrator

Một lệnh `run(query)` → Planning → Routing → Execution. Tự route QA hoặc Prediction.

In [ ]:
from calm.orchestrator import CALMOrchestrator
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.web_search import WebSearchTool

memory_store = ChromaMemoryStore(collection_name="calm_orch", persist_directory=".chroma")
tools = {"web_search": WebSearchTool(safety_checker=SafetyChecker(llm=llm), config={"max_news_results": 3})}
orch = CALMOrchestrator.from_llm(llm=llm, memory_store=memory_store, tools=tools, config={"planner_n_max": 2})

# Thử QA hoặc Prediction: orch.run("Predict wildfire risk California next 7 days")
result = orch.run("What causes wildfires in the Amazon?")
print("Task type:", result.get("task_type"))
if result.get("task_type") == "qa":
    print("Answer:", (result.get("answer", "") or "")[:250])
else:
    print("Risk:", result.get("risk_level"), "|", result.get("decision"), "|", (result.get("rationale", "") or "")[:150])
if result.get("error"):
    print("Error:", str(result["error"])[:150])

## 8. Evaluator — LLM-as-a-Judge

Chấm 5 tiêu chí (0–100): Data Accuracy, Explainability, Jargon Avoidance, Redundancy Avoidance, Citation Quality. Passing: average >= 75.

In [ ]:
from calm.agents.evaluator_agent import EvaluatorAgent

evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})
eval_query = "What causes wildfires in the Amazon?"
orch_result = orch.run(eval_query)  # hoặc dùng result từ cell trên
eval_result = evaluator.evaluate(response=orch_result, query=eval_query)

print("Average score:", eval_result.get("average_score"))
print("Passed:", eval_result.get("passed"))
print("Scores:", eval_result.get("scores", {}))